# Immortalite — Self-Play Training (Colab)

Trains the lightweight self-play chess engine on a free GPU. Checkpoints are written to Google Drive so the run survives disconnects.

**Steps:** set runtime to GPU (Runtime -> Change runtime type -> GPU), then run the cells top to bottom.

In [ ]:
# 1. Get the code. Safe to re-run (always works from /content, never nests).
import os
%cd /content
if not os.path.exists('/content/immortalite'):
    !git clone https://github.com/carlo-wong/immortalite.git
%cd /content/immortalite
!git pull --quiet
print('working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies (Colab already has a CUDA torch build).
!pip install -q python-chess numpy tqdm

In [ ]:
# 3. Mount Google Drive for crash-proof checkpoints.
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/immortalite_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

In [ ]:
# 4. Confirm GPU is available.
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 5. Train with staged C1/C2 controls.
# - C1 ramp: choose one of off/24/32/48/64.
# - C2 shaping: optional max-move cap + draw penalty.
# - Resignation: optional follow-up once C2 shaping is stable.
import os, torch

resume = os.path.join(CKPT_DIR, 'latest.pt')
resume_arg = f'--resume {resume}' if os.path.exists(resume) else ''

if torch.cuda.is_available():
    print('GPU detected -> --device cuda --gpu')
    preset = '--device cuda --gpu'
else:
    print('No GPU (quota exhausted?) -> --device cpu --light. Training will be slow.')
    preset = '--device cpu --light'

# ---- C1 throughput ramp ----
# off (baseline), then 24, 32, 48, 64.
C1_STAGE = '24'

# ---- C2 shaping (toggle once C1 is stable) ----
ENABLE_C2 = False
C2_MAX_GAME_MOVES = 140
C2_DRAW_PENALTY = 0.15

# ---- Optional resignation (enable only after C2 shaping is stable) ----
ENABLE_RESIGN = False
RESIGN_THRESHOLD = -0.90
RESIGN_PLIES = 3
RESIGN_MIN_MOVES = 20

c1_arg = f'--c1-stage {C1_STAGE}' if C1_STAGE != 'off' else ''
c2_arg = ''
if ENABLE_C2:
    c2_arg = f'--max-game-moves {C2_MAX_GAME_MOVES} --draw-penalty {C2_DRAW_PENALTY}'
resign_arg = ''
if ENABLE_RESIGN:
    resign_arg = (
        f'--resign-threshold {RESIGN_THRESHOLD} '
        f'--resign-plies {RESIGN_PLIES} '
        f'--resign-min-moves {RESIGN_MIN_MOVES}'
    )

!python -m engine.train --iterations 50 {preset} {c1_arg} {c2_arg} {resign_arg} --save-every 10 --gate-every 0 --gate-games 20 --checkpoint-dir "$CKPT_DIR" $resume_arg

In [ ]:
# 6. Independent Strength Gating (Checkpoints A vs B)
# Compare any two checkpoints (e.g., 'latest' or iteration numbers like 40, 50).
# This runs independently of the training loop and survives disconnects.
import os
import torch
import numpy as np
from engine.config import Config
from engine.network import ChessNet
from engine.train import play_match

# ---- Gating Match Configuration ----
CHECKPOINT_A = 'latest'  # Can be 'latest' or an iteration number (e.g., 50)
CHECKPOINT_B = 40        # Can be 'latest' or an iteration number (e.g., 40)
GAMES = 20               # Number of games to play in the match
SIMS = 128               # Simulations per move
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# ------------------------------------

cfg = Config()

# Resolve Path A
if CHECKPOINT_A == 'latest':
    path_a = os.path.join(CKPT_DIR, 'latest.pt')
    label_a = "Latest"
else:
    path_a = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_A:04d}.pt')
    label_a = f"Iter {CHECKPOINT_A}"

# Resolve Path B
if CHECKPOINT_B == 'latest':
    path_b = os.path.join(CKPT_DIR, 'latest.pt')
    label_b = "Latest"
else:
    path_b = os.path.join(CKPT_DIR, f'ckpt_iter_{CHECKPOINT_B:04d}.pt')
    label_b = f"Iter {CHECKPOINT_B}"

if not os.path.exists(path_a) or not os.path.exists(path_b):
    print(f"Error: Make sure both checkpoints exist!\nPath A: {path_a}\nPath B: {path_b}")
else:
    print(f"Loading Checkpoint A ({label_a}) from {path_a}...")
    state_a = torch.load(path_a, map_location=DEVICE)
    net_a = ChessNet(cfg.net).to(DEVICE)
    net_a.load_state_dict(state_a['model'] if isinstance(state_a, dict) and 'model' in state_a else state_a, strict=False)
    net_a.eval()

    print(f"Loading Checkpoint B ({label_b}) from {path_b}...")
    state_b = torch.load(path_b, map_location=DEVICE)
    net_b = ChessNet(cfg.net).to(DEVICE)
    net_b.load_state_dict(state_b['model'] if isinstance(state_b, dict) and 'model' in state_b else state_b, strict=False)
    net_b.eval()

    print(f"\nStarting Match: {label_a} vs {label_b} ({GAMES} games, {SIMS} sims, on {DEVICE})...")
    
    winrate = play_match(net_a, net_b, cfg, n_games=GAMES, sims=SIMS, device=DEVICE)
    
    print("\n" + "="*40)
    print(f"MATCH COMPLETED!")
    print(f"{label_a} score vs {label_b}: {winrate:.3f}")
    if winrate > 0.55:
        print(f"Result: {label_a} PASSED the gate (Stronger than {label_b})!")
    elif winrate < 0.45:
        print(f"Result: {label_a} is significantly WEAKER than {label_b}!")
    else:
        print(f"Result: {label_a} and {label_b} are roughly EQUAL (within noise margin).")
    print("="*40)


In [ ]:
# 7. Plot training progress (run AFTER at least one training iteration finishes).
import os
import pandas as pd
import matplotlib.pyplot as plt

path = os.path.join(CKPT_DIR, 'metrics.csv')
if not os.path.exists(path):
    print('No metrics.csv yet. Wait for iteration 0 of cell 5 to finish '
          '(a few minutes on the --gpu preset), then re-run this cell.')
else:
    df = pd.read_csv(path)
    if df.empty:
        print('metrics.csv exists but is empty — wait for an iteration to finish.')
    else:
        df['iter'] = pd.to_numeric(df['iter'], errors='coerce')
        df = df.dropna(subset=['iter']).copy()
        if df.empty:
            print('metrics.csv has no valid iteration rows yet.')
        else:
            for col in df.columns:
                if col != 'terminations':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            fig, ax = plt.subplots(2, 3, figsize=(16, 8))
            ax = ax.flatten()

            def plot_panel(axis, cols, title, ylim=None):
                present = [c for c in cols if c in df.columns]
                if not present:
                    axis.text(0.5, 0.5, 'missing columns', ha='center', va='center')
                    axis.set_title(title)
                    axis.set_xlabel('iteration')
                    return
                df.plot(x='iter', y=present, ax=axis, title=title)
                axis.set_xlabel('iteration')
                if ylim is not None:
                    axis.set_ylim(*ylim)

            plot_panel(ax[0], ['policy_loss', 'value_loss'], 'Losses')
            plot_panel(ax[1], ['policy_entropy', 'grad_norm'], 'Entropy / Gradient norm')
            plot_panel(ax[2], ['value_sign_acc', 'policy_top1_agree'], 'Net target agreement', ylim=(0.0, 1.0))
            plot_panel(ax[3], ['decisive_rate', 'draw_rate', 'max_moves_trunc_rate'], 'Outcome mix', ylim=(0.0, 1.0))
            plot_panel(ax[4], ['mean_game_len'], 'Mean game length')

            if 'winrate_vs_prev' not in df.columns:
                ax[5].text(0.5, 0.5, 'winrate_vs_prev missing', ha='center', va='center')
                ax[5].set_title('Strength gate (vs prev net)')
                ax[5].set_xlabel('iteration')
            else:
                gate_df = df.dropna(subset=['winrate_vs_prev'])
                if gate_df.empty:
                    ax[5].text(0.5, 0.5, 'no gate rows yet', ha='center', va='center')
                    ax[5].set_title('Strength gate (vs prev net)')
                    ax[5].set_xlabel('iteration')
                else:
                    gate_df.plot(x='iter', y='winrate_vs_prev', marker='o', ax=ax[5], title='Strength gate (vs prev net)')
                    ax[5].set_xlabel('iteration')
                ax[5].axhline(0.5, color='gray', linestyle='--', linewidth=1)
                ax[5].set_ylim(0.0, 1.0)

            plt.tight_layout()
            plt.show()

## After training
Download `latest.pt` from your Drive folder and run the analysis server locally:

```bash
set IMMORTALITE_CHECKPOINT=path\to\latest.pt   # Windows
python -m uvicorn server.app:app --port 8000
```
Then open http://localhost:8000/app/ .